# Project Journal 06: JAX Wave Lab

This entry documents the first JAX sidecar in the project: a small CPU wave-propagation sandbox that is separate from the main PyTorch monitoring pipeline but directly useful for PhD-style skill building.


## Context

Up to this point, the project was strong on monitoring and field evaluation but weak on explicit wave-propagation tooling.

The goal of this stage was not to rewrite the repo in JAX. The goal was to create one concrete JAX lab where you can learn:

- JIT-compiled wavefield rollout
- batched simulation with `vmap`
- gradient computation through propagation
- runtime differences between NumPy, JAX eager, and JAX JIT on CPU


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

ROOT = Path.cwd().resolve()
if not (ROOT / 'runs').exists():
    ROOT = ROOT.parent.resolve()

summary = json.loads((ROOT / 'runs' / 'jax_wave_lab' / 'results' / 'jax_summary.json').read_text())
runtime_df = pd.read_csv(ROOT / 'runs' / 'jax_wave_lab' / 'results' / 'jax_runtime_table.csv')
ROOT


## What Changed This Stage

The JAX lab adds a new sidecar capability rather than changing the core paper pipeline:

- a small 2D acoustic wave-propagation rollout
- a batched `vmap` benchmark
- a gradient sanity check with respect to the velocity model
- saved visual outputs and a runtime table

That makes the project feel more like a real geophysics-and-ML research codebase and less like a pure segmentation repo.


In [ ]:
display(pd.DataFrame([summary]))
display(runtime_df)


## Current Results

The most useful outcome here is that the JAX path is real and numerically sane:

- the JAX and NumPy wavefields match to a very small maximum absolute error
- the first JIT call is slower because of compilation
- the cached JIT call is much faster than JAX eager execution on CPU
- the lab now produces wavefield snapshots, a GIF, and a gradient image that you can actually inspect

This is the kind of sidecar that can grow into a later PhD chapter or second paper without destabilizing the monitoring baseline.


In [ ]:
snapshot_path = ROOT / 'runs' / 'jax_wave_lab' / 'results' / 'figures' / 'jax_wavefield_snapshots.png'
gradient_path = ROOT / 'runs' / 'jax_wave_lab' / 'results' / 'figures' / 'jax_gradient_sanity.png'

snapshot_image = mpimg.imread(snapshot_path)
plt.figure(figsize=(14, 4))
plt.imshow(snapshot_image)
plt.title('JAX wavefield snapshots')
plt.axis('off')
plt.show()

gradient_image = mpimg.imread(gradient_path)
plt.figure(figsize=(6, 4))
plt.imshow(gradient_image)
plt.title('JAX gradient sanity plot')
plt.axis('off')
plt.show()

Image(filename=str(ROOT / 'runs' / 'jax_wave_lab' / 'results' / 'figures' / 'jax_wavefield_animation.gif'))


## Open Problems

This wave lab is intentionally small, so there are still real limits:

- it is a 2D toy forward model, not a production viscoacoustic solver
- it currently runs on CPU only in this environment
- it is not yet connected to the main CCS monitoring task or to the field data stack

That is okay for now. Its purpose is to build intuition, prove the JAX path is viable, and give you a safe place to learn wave-physics ideas without breaking the main paper track.


In [ ]:
runtime_df.assign(speedup_vs_eager=runtime_df['runtime_seconds'].iloc[1] / runtime_df['runtime_seconds'])


## Next Stage

The natural next move for the JAX sidecar is one of these:

1. add receivers and seismogram-style outputs,
2. add a simple inversion or gradient-descent toy experiment, or
3. scale the same sandbox to a small cluster-ready benchmark.

For the main project, though, the biggest immediate win is that JAX is now a real, runnable part of the repo instead of a future aspiration.
